In [107]:
import litellm
import os
from dotenv import load_dotenv
import fitz  # PyMuPDF, per lavorare con i PDF
import numpy as np
import instructor
from pydantic import BaseModel, Field

In [108]:
load_dotenv()

True

In [109]:



class BookSummarizer:

    summary_prompt = """

    Il tuo compito è di riassumere la sezione del libro che ti viene fornita.
    Fornisci un riassunti chiaro e conciso, ma riporta tutte le informazioni contenute nel passaggio.
    Se la sezione contiene esercizi o esempi o informazioni non strettamente necessarie alla comprensione del testo, puoi ometterli. Concentrati sui concetti chiave e sulle informazioni essenziali.
    Se il testo contiene equazioni, formule o dimostrazioni, riportale in modo chiaro e preciso nel riassunto.

    Testo da riassumere:    {section_title}
    {section_text}

    """

    grouping_prompt = """
    Il tuo compito è quello di raggruppare questi riassunti di sezioni di un libro in base ai concetti che contengono.
    Riceverai una lista di riassunti, ognuno associato a un titolo di sezione. Analizzali, estrai i concetti chiave e restituisci
    in formato json una lista di titoli di gruppi di concetti, ognuno associato a una lista di indici dei riassunti che contengono quei concetti.
    Restituisci solo il json, senza spiegazioni o commenti.
    Non suddividere eccessivamente i gruppi, cerca di raggruppare insieme concetti simili o correlati, ma non creare gruppi troppo ampi.
    Escludi dai gruppi riferimenti bibliografici, esempi, esercizi o informazioni non strettamente necessarie alla comprensione dei concetti.

    Testo dei riassunti:
    {summary_list_text}

    """

    write_notes_prompt = """
    Il tuo compito è quello di scrivere degli appunti chiari, sintetici e dettagliati a partire da un gruppo di riassunti di sezioni di un libro che trattano concetti simili.
    Riceverai un titolo di un gruppo di concetti e una lista di riassunti associati a quel gruppo.
    Analizza i riassunti, estrai i concetti chiave e scrivi degli appunti che sintetizzino in modo chiaro e dettagliato i concetti trattati nei riassunti.
    Restituisci solo il testo degli appunti in formato markdown.
    Titolo del gruppo di concetti: {group_title}
    Riassunti associati:
    {summary_list_text}

    """

    add_links_prompt = """
    Il tuo compito è quello di aggiungere link [[wikistyle]] ad un file di appunti in markdown, basandoti su un indice.
    I link devono essere aggiunti solo quando un concetto menzionato negli appunti è presente nell'indice, e devono essere aggiunti solo alla prima occorrenza del concetto.
    Non aggiungere link a documenti non presenti nell'indice fornito.
    Riceverai un testo in markdown e un indice di concetti. Analizza il testo, individua i concetti presenti nell'indice e aggiungi i link [[wikistyle]]
    al testo. Restituisci solo il testo con i link aggiunti, senza spiegazioni o commenti.

        Testo degli appunti:
        {notes_text}

        Indice dei concetti:
        {concept_index_text}

    """

    class GroupDataModel(BaseModel):
        group_title: str = Field(description="Titolo conciso del concetto o argomento principale che accomuna i riassunti.")
        summary_indices: list[int] = Field(description="Lista degli indici dei riassunti che trattano questo concetto.")

    class GroupingResult(BaseModel):
        groups: list['BookSummarizer.GroupDataModel'] = Field(description="Lista dei gruppi di concetti individuati.")

    models = {
        "haiku" : "claude-haiku-4-5",
        "sonnet" : "claude-sonnet-4-6",
        "opus" : "claude-opus-4-8"
    }

    def __init__(self, pdf_path):
        self.pdf_path = pdf_path
        self.doc = fitz.open(pdf_path)
        self.toc = None
        self.sections = []
        self.summaries = {}
        self.grouping_result = None
        self.notes_by_group = {}
        self.finished_notes = {}

    def extract_toc(self):
        self.toc = self.doc.get_toc()
        if not self.toc:
            raise ValueError("No Table of Contents found in the PDF.")
        return self.toc
    
    def split_into_sections(self, force=False):
        if self.sections and not force:
            return self.sections

        if self.toc is None:
            self.extract_toc()
        
        section_starts = np.array([item[2] for item in self.toc])
        section_ends = np.append(section_starts[1:] - 1, self.doc.page_count)
        section_names = [item[1] for item in self.toc]
        
        for i, (start, end) in enumerate(zip(section_starts, section_ends)):
            section_data = (section_names[i], start, end)
            self.sections.append(section_data)
        
        return self.sections
    
    def extract_section_text(self, start_page, end_page):
        text = ""
        for page_num in range(start_page, end_page):
            page = self.doc.load_page(page_num)
            text += page.get_text()
        return text
    
    def get_section_text(self, section_index):

        if not self.sections:
            self.split_into_sections()

        if section_index < 0 or section_index >= len(self.sections):
            raise IndexError("Section index out of range.")
        
        section_name, start_page, end_page = self.sections[section_index]
        return section_name, self.extract_section_text(start_page, end_page)

    def summarize_section(self, section_index, model="haiku"):

        if model not in self.models:
            raise ValueError(f"Model '{model}' not found. Available models: {list(self.models.keys())}")

        section_name, section_text = self.get_section_text(section_index)

        completed_prompt = self.summary_prompt.format(section_title=section_name, section_text=section_text)

        response = litellm.completion(
            model=self.models[model],
            messages=[{"role": "user", "content": completed_prompt}]
        )

        summary = response.choices[0].message.content

        self.summaries[section_index] = summary

        return summary

    def summarize_all_sections(self, model="haiku", section_indices=None):
        if section_indices is None:
            section_indices = range(len(self.sections))

        for i in section_indices:
            self.summarize_section(i, model=model)

    def get_summary(self, section_index):
        return self.summaries.get(section_index, None)
    
    def get_all_summaries(self):
        return self.summaries

    def save_summaries(self, output_dir):
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)

        for index, summary in self.summaries.items():
            section_name = self.sections[index][0].replace(" ", "_")
            output_path = os.path.join(output_dir, f"{section_name}_summary.txt")
            with open(output_path, "w") as f:
                f.write(summary)

    def group_summaries(self, model="haiku"):
        if model not in self.models:
            raise ValueError(f"Model '{model}' not found. Available models: {list(self.models.keys())}")

        if not self.summaries:
            raise ValueError("No summaries available to group. Please summarize sections first.")

        client = instructor.from_litellm(litellm.completion)

        summary_list_text = ""
        for index, summary in self.summaries.items():
            titolo_sezione = self.sections[index][0]
            summary_list_text += f"--- INDICE {index} | SEZIONE: {titolo_sezione} ---\n{summary}\n\n"

        completed_prompt = self.grouping_prompt.format(summary_list_text=summary_list_text)
        
        response = client.chat.completions.create(
            model=self.models[model],
            response_model=self.GroupingResult,
            messages=[
                {"role": "system", "content": "Sei un assistente accademico specializzato nell'estrarre e raggruppare concetti chiave."},
                {"role": "user", "content": completed_prompt}
            ],
            temperature=0.2
        )

        self.grouping_result = response

        #self.grouping_result = response.choices[0].message.content

        return response

    def write_notes_for_group(self, group_title, summary_list_text, model="haiku"):
        if model not in self.models:
            raise ValueError(f"Model '{model}' not found. Available models: {list(self.models.keys())}")

        completed_prompt = self.write_notes_prompt.format(group_title=group_title, summary_list_text=summary_list_text)

        response = litellm.completion(
            model=self.models[model],
            messages=[{"role": "user", "content": completed_prompt}]
        )

        notes = response.choices[0].message.content

        self.notes_by_group[group_title] = notes

        return notes
    
    def write_notes_for_all_groups(self, model="haiku"):
        if self.grouping_result is None:
            raise ValueError("No grouping result available. Please run group_summaries first.")

        notes_by_group = {}

        for group in self.grouping_result.groups:
            group_title = group.group_title
            summary_indices = group.summary_indices

            summary_list_text = ""
            for index in summary_indices:
                titolo_sezione = self.sections[index][0]
                summary = self.summaries[index]
                summary_list_text += f"--- INDICE {index} | SEZIONE: {titolo_sezione} ---\n{summary}\n\n"

            notes = self.write_notes_for_group(group_title, summary_list_text, model=model)
            notes_by_group[group_title] = notes
        
        return notes_by_group
    
    def save_notes(self, output_dir):
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)

        for group_title, notes in self.notes_by_group.items():
            safe_group_title = group_title.replace(" ", "_")
            output_path = os.path.join(output_dir, f"{safe_group_title}.md")
            with open(output_path, "w") as f:
                f.write(notes)

    def add_links_to_notes(self, model="haiku"):

        index = self.notes_by_group.keys()
        notes_texts = self.notes_by_group.values()
        processed_notes = []

        for note in notes_texts:
            completed_prompt = self.add_links_prompt.format(notes_text=note, concept_index_text="\n".join(index).replace("_", " "))

            response = litellm.completion(
                model=self.models[model],
                messages=[{"role": "user", "content": completed_prompt}]
            )

            linked_note = response.choices[0].message.content

            processed_notes.append(linked_note)
        
        self.finished_notes = {group_title: linked_note for group_title, linked_note in zip(index, processed_notes)}
        
        return processed_notes

    def save_finished_notes(self, output_dir):
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)

        for group_title, notes in self.finished_notes.items():
            safe_group_title = group_title.replace(" ", "_")
            output_path = os.path.join(output_dir, f"{safe_group_title}.md")
            with open(output_path, "w") as f:
                f.write(notes)


In [110]:
pdf_path = "data/luth_full.pdf"
summarizer = BookSummarizer(pdf_path)

In [111]:
summarizer.summarize_all_sections(model="sonnet", section_indices=list(range(4, 40)))

In [112]:
for title, summary in summarizer.get_all_summaries().items():
    print(f"--- SEZIONE {title} ---\n{summary}\n\n")

--- SEZIONE 4 ---
# Riassunto – Capitolo 1: Fisica delle Superfici e delle Interfacce: Definizione e Importanza

## Limiti della fisica dello stato solido classica

La fisica dello stato solido classica tratta i cristalli come oggetti di estensione infinita, sfruttando la simmetria traslazionale per semplificare il trattamento matematico. Questa approssimazione è valida per:
- Proprietà macroscopiche dipendenti dal numero totale di atomi
- Esperimenti spettroscopici con sonde penetranti (raggi X, neutroni, elettroni veloci) che interagiscono con il bulk e rendono trascurabile il contributo degli atomi superficiali (~10¹⁵ cm⁻²)

Tuttavia, questo approccio diventa **inadeguato** quando si utilizzano sonde che interagiscono fortemente con la materia e penetrano solo pochi Ångström (elettroni a bassa energia, fasci atomici e molecolari). In questi casi le proprietà degli atomi superficiali — diverse da quelle degli atomi nel bulk — diventano determinanti. Analogamente, nella **fotoemission

In [113]:
grouping_result = summarizer.group_summaries(model="sonnet")

In [114]:
for group in grouping_result.groups:
    print(f"--- GRUPPO: {group.group_title} ---")
    print(f"Riassunti associati: {group.summary_indices}\n")

--- GRUPPO: Introduzione alla Fisica delle Superfici e delle Interfacce ---
Riassunti associati: [4]

--- GRUPPO: Tecnologia dell'Ultravuoto (UHV) e Misura della Pressione ---
Riassunti associati: [5, 11]

--- GRUPPO: Ottica delle Particelle, Lenti e Analizzatori di Energia ---
Riassunti associati: [7]

--- GRUPPO: Preparazione di Superfici: Clivaggio, Bombardamento Ionico e Ricottura ---
Riassunti associati: [12, 13]

--- GRUPPO: Crescita Epitassiale: MBE e MOMBE ---
Riassunti associati: [14, 15]

--- GRUPPO: Tecniche Analitiche Superficiali: AES, SIMS e SEM ---
Riassunti associati: [16, 18, 32]

--- GRUPPO: Stress, Energia Superficiale e Forma di Equilibrio dei Cristalli ---
Riassunti associati: [22]

--- GRUPPO: Rilassamento, Ricostruzione e Difetti delle Superfici ---
Riassunti associati: [23]

--- GRUPPO: Reticoli 2D, Sovrastrutture e Spazio Reciproco ---
Riassunti associati: [24, 25, 26]

--- GRUPPO: Struttura delle Interfacce Solido-Solido e Mismatch Reticolare ---
Riassunti ass

In [115]:
summarizer.write_notes_for_all_groups(model="haiku")
summarizer.save_notes(output_dir="output/notes")

In [116]:
summarizer.add_links_to_notes(model="haiku")
summarizer.save_finished_notes(output_dir="output/linked_notes")